## Different levels of codeswitching for SVA task

In [ ]:
import json
import numpy as np
import random
import time
import os

# === Logger helper ===
def log_time(msg):
    print(f"[{time.strftime('%Y-%m-%d %H:%M:%S')}] {msg}")

# === Define nouns and adjective pairs separately ===
def create_french_gender_dataset():
    nouns = [
        "professeur", "directeur", "acteur", "étudiant", "docteur",
        "artiste", "ami", "avocat", "ingénieur", "serveur"
    ]

    adjective_pairs = [
        ("intelligent", "intelligente"),
        ("heureux", "heureuse"),
        ("talentueux", "talentueuse"),
        ("studieux", "studieuse"),
        ("compétent", "compétente"),
        ("créatif", "créative"),
        ("gentil", "gentille"),
        ("persuasif", "persuasive"),
        ("efficace", "efficace"),  # adj identical in gender
        ("rapide", "rapide")        # adj identical in gender
    ]

    examples_dict = {}

    log_time("Starting dataset generation...")

    idx = 0
    for noun in nouns:
        for adj_masc, adj_fem in adjective_pairs:
            examples_dict[idx] = {
                "masc_sentence": f"Le {noun} {adj_masc}",
                "fem_sentence": f"La {noun} {adj_fem}",
                "masc_label": adj_masc,
                "fem_label": adj_fem
            }
            idx += 1

    log_time("Dataset generation complete.")

    # Save the dataset
    output_dir = './datasets/gender_agreement/'
    os.makedirs(output_dir, exist_ok=True)
    output_path = f"{output_dir}/gender_agreement_dataset.json"

    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump({"examples": examples_dict}, f, indent=4, ensure_ascii=False)

    log_time(f"Dataset saved to {output_path}")

# === Execute ===
if __name__ == "__main__":
    random.seed(42)
    np.random.seed(42)
    create_french_gender_dataset()

## Codeswitched

In [6]:
import os
import json
import numpy as np
import random

def create_codeswitched_sva_dataset(noun_list_tuples, verb_list_tuples, pred_verb_tuples, 
                                    codeswitch_level='no_switching', random_seed=42):

    np.random.seed(random_seed)
    random.seed(random_seed)

    noun_sing_es = [f'{noun[0]}' for noun in noun_list_tuples]
    noun_plur_es = [f'{noun[1]}' for noun in noun_list_tuples]

    verb_main_sing_es = [f'{verb[0]}' for verb in verb_list_tuples]
    verb_main_plur_es = [f'{verb[1]}' for verb in verb_list_tuples]

    verb_pred_sing_es = [f'{verb[0]}' for verb in pred_verb_tuples]
    verb_pred_plur_es = [f'{verb[1]}' for verb in pred_verb_tuples]

    noun_translations = {
        "pintor": "painter", "pintores": "painters",
        "ingeniero": "engineer", "ingenieros": "engineers",
        "detective": "detective", "detectives": "detectives",
        "ministro": "minister", "ministros": "ministers",
        "maestro": "teacher", "maestros": "teachers"
    }

    verb_translations = {
        "encontró": "found", "encontraron": "found",
        "puso": "put", "pusieron": "put",
        "tuvo": "had", "tuvieron": "had",
        "tiene": "has", "tienen": "have",
        "dio": "gave", "dieron": "gave"
    }

    dataset = {}

    idx = 0
    for i, noun1 in enumerate(noun_sing_es):
        for j, verb in enumerate(verb_main_sing_es):
            for k, noun2 in enumerate(noun_sing_es):
                if i == k:
                    continue

                base_singular = f"El {noun1} que {verb_main_sing_es[j]} al {noun2}"
                base_plural = f"Los {noun_plur_es[i]} que {verb_main_plur_es[j]} al {noun2}"
                label_sing = verb_pred_sing_es[j]
                label_plur = verb_pred_plur_es[j]

                def codeswitch(sentence, level):
                    words = sentence.split()
                    new_words = []
                    for word in words:
                        word_clean = word.lower().strip(".,!?")
                        if level in ['noun_switching', 'full_switching'] and word_clean in noun_translations:
                            new_words.append(noun_translations[word_clean])
                        elif level in ['verb_switching', 'full_switching'] and word_clean in verb_translations:
                            new_words.append(verb_translations[word_clean])
                        else:
                            new_words.append(word)
                    return ' '.join(new_words)

                if codeswitch_level != 'no_switching':
                    base_singular = codeswitch(base_singular, codeswitch_level)
                    base_plural = codeswitch(base_plural, codeswitch_level)

                dataset[idx] = {
                    "src": base_plural,
                    "base": base_singular,
                    "src_label": label_plur,
                    "base_label": label_sing,
                    "base_subject_number": "singular"
                }
                idx += 1

                dataset[idx] = {
                    "src": base_singular,
                    "base": base_plural,
                    "src_label": label_sing,
                    "base_label": label_plur,
                    "base_subject_number": "plural"
                }
                idx += 1

    keys = list(dataset.keys())
    random.shuffle(keys)
    dataset_shuffled = {new_idx: dataset[k] for new_idx, k in enumerate(keys)}

    output_dir = './new_exp'
    os.makedirs(output_dir, exist_ok=True)

    filename = f"{output_dir}/spanish_{codeswitch_level}_sva.json"
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(dataset_shuffled, f, ensure_ascii=False, indent=4)
    print(f"Saved dataset at: {filename}")

if __name__ == "__main__":
    noun_list_tuples = [("pintor", "pintores"), ("ingeniero", "ingenieros"),
                        ("detective", "detectives"), ("ministro", "ministros"), ("maestro", "maestros")]
    verb_list_tuples = [("encontró", "encontraron"), ("puso", "pusieron"), ("dio", "dieron")]
    pred_verb_tuples = [("tiene", "tienen"), ("tuvo", "tuvieron"), ("dio", "dieron")]

    levels = ['no_switching', 'noun_switching', 'verb_switching', 'full_switching']
    for level in levels:
        create_codeswitched_sva_dataset(noun_list_tuples, verb_list_tuples, pred_verb_tuples, 
                                        codeswitch_level=level)


Saved dataset at: ./new_exp/spanish_no_switching_sva.json
Saved dataset at: ./new_exp/spanish_noun_switching_sva.json
Saved dataset at: ./new_exp/spanish_verb_switching_sva.json
Saved dataset at: ./new_exp/spanish_full_switching_sva.json


## Codeswitched: corrupted.

In [5]:
import os
import json
import numpy as np
import random

def create_corrupted_codeswitched_sva_dataset(noun_list_tuples, verb_list_tuples, pred_verb_tuples, 
                                             codeswitch_level='no_switching', random_seed=42):
    """
    Create a grammatically disrupted code-switched SVA dataset with consistent code-switching
    but broken grammar to produce jumbled PCA plots.
    
    Each code-switching level maintains consistent word replacement patterns:
    - no_switching: No words are replaced
    - noun_switching: Only nouns are consistently switched to English
    - verb_switching: Only verbs are consistently switched to English
    - full_switching: Both nouns and verbs are consistently switched to English
    
    The grammar disruption is applied separately from code-switching, through:
    1. Mismatching determiners with nouns (El with plural nouns, Los with singular nouns)
    2. Using incorrect verb forms (singular verbs with plural subjects and vice versa)
    3. Randomly swapping the verb label (prediction)
    """
    np.random.seed(random_seed)
    random.seed(random_seed)

    noun_sing_es = [f'{noun[0]}' for noun in noun_list_tuples]
    noun_plur_es = [f'{noun[1]}' for noun in noun_list_tuples]

    verb_main_sing_es = [f'{verb[0]}' for verb in verb_list_tuples]
    verb_main_plur_es = [f'{verb[1]}' for verb in verb_list_tuples]

    verb_pred_sing_es = [f'{verb[0]}' for verb in pred_verb_tuples]
    verb_pred_plur_es = [f'{verb[1]}' for verb in pred_verb_tuples]

    noun_translations = {
        "pintor": "painter", "pintores": "painters",
        "ingeniero": "engineer", "ingenieros": "engineers",
        "detective": "detective", "detectives": "detectives",
        "ministro": "minister", "ministros": "ministers",
        "maestro": "teacher", "maestros": "teachers"
    }

    verb_translations = {
        "encontró": "found", "encontraron": "found",
        "puso": "put", "pusieron": "put",
        "tuvo": "had", "tuvieron": "had",
        "tiene": "has", "tienen": "have",
        "dio": "gave", "dieron": "gave"
    }

    # Create jumbled determiners that mismatch the noun number
    determiners = {
        "singular_correct": "El",
        "singular_incorrect": "Los",
        "plural_correct": "Los", 
        "plural_incorrect": "El"
    }

    dataset = {}
    idx = 0
    
    for i, noun1 in enumerate(noun_sing_es):
        for j, verb in enumerate(verb_main_sing_es):
            for k, noun2 in enumerate(noun_sing_es):
                if i == k:
                    continue

                # First, create normal sentences
                base_singular = f"El {noun1} que {verb_main_sing_es[j]} al {noun2}"
                base_plural = f"Los {noun_plur_es[i]} que {verb_main_plur_es[j]} al {noun2}"
                label_sing = verb_pred_sing_es[j] 
                label_plur = verb_pred_plur_es[j]

                # Define code-switching function that preserves consistency based on level
                def codeswitch(sentence, level):
                    words = sentence.split()
                    new_words = []
                    for word in words:
                        word_clean = word.lower().strip(".,!?")
                        if level in ['noun_switching', 'full_switching'] and word_clean in noun_translations:
                            new_words.append(noun_translations[word_clean])
                        elif level in ['verb_switching', 'full_switching'] and word_clean in verb_translations:
                            new_words.append(verb_translations[word_clean])
                        else:
                            new_words.append(word)
                    return ' '.join(new_words)

                # Apply consistent code-switching
                if codeswitch_level != 'no_switching':
                    base_singular = codeswitch(base_singular, codeswitch_level)
                    base_plural = codeswitch(base_plural, codeswitch_level)

                # Create grammatically broken sentences (independently of code-switching):
                
                # 1. Randomly disrupt determiner-noun agreement (40% of cases)
                def disrupt_determiner_noun_agreement(sentence, is_singular):
                    if random.random() < 0.5:
                        words = sentence.split()
                        if is_singular and words[0] == "El":
                            words[0] = determiners["singular_incorrect"]
                        elif not is_singular and words[0] == "Los":
                            words[0] = determiners["plural_incorrect"]
                        return ' '.join(words)
                    return sentence

                # 2. Randomly swap verb forms (40% of cases)
                def disrupt_verb_agreement(sentence, verb_sing, verb_plur, is_sing_sentence):
                    if random.random() < 0.5:
                        # Find which verb form appears in the sentence
                        if verb_sing in sentence and is_sing_sentence:
                            return sentence.replace(verb_sing, verb_plur)
                        elif verb_plur in sentence and not is_sing_sentence:
                            return sentence.replace(verb_plur, verb_sing)
                    return sentence

                # Apply disruptions to both singular and plural sentences
                corrupt_base_singular = disrupt_determiner_noun_agreement(base_singular, True)
                corrupt_base_plural = disrupt_determiner_noun_agreement(base_plural, False)
                
                corrupt_base_singular = disrupt_verb_agreement(
                    corrupt_base_singular, 
                    verb_main_sing_es[j], 
                    verb_main_plur_es[j], 
                    True
                )
                corrupt_base_plural = disrupt_verb_agreement(
                    corrupt_base_plural, 
                    verb_main_sing_es[j], 
                    verb_main_plur_es[j], 
                    False
                )

                # 3. Randomly swap the verb label (prediction) (40% of cases)
                if random.random() < 0.7:
                    temp = label_sing
                    label_sing = label_plur
                    label_plur = temp

                # Create dataset entries with the corrupted sentences
                dataset[idx] = {
                    "src": corrupt_base_plural,
                    "base": corrupt_base_singular,
                    "src_label": label_plur,
                    "base_label": label_sing,
                    "base_subject_number": "singular"
                }
                idx += 1

                dataset[idx] = {
                    "src": corrupt_base_singular,
                    "base": corrupt_base_plural,
                    "src_label": label_sing,
                    "base_label": label_plur,
                    "base_subject_number": "plural"
                }
                idx += 1

    # Shuffle the dataset
    keys = list(dataset.keys())
    random.shuffle(keys)
    dataset_shuffled = {new_idx: dataset[k] for new_idx, k in enumerate(keys)}

    # Save the corrupted dataset
    output_dir = './corrupted_cs_datasets'
    os.makedirs(output_dir, exist_ok=True)

    filename = f"{output_dir}/corrupted_spanish_{codeswitch_level}_sva.json"
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(dataset_shuffled, f, ensure_ascii=False, indent=4)
    print(f"Saved corrupted dataset at: {filename}")

if __name__ == "__main__":
    noun_list_tuples = [("pintor", "pintores"), ("ingeniero", "ingenieros"),
                         ("detective", "detectives"), ("ministro", "ministros"), ("maestro", "maestros")]
    verb_list_tuples = [("encontró", "encontraron"), ("puso", "pusieron"), ("dio", "dieron")]
    pred_verb_tuples = [("tiene", "tienen"), ("tuvo", "tuvieron"), ("dio", "dieron")]

    levels = ['no_switching', 'noun_switching', 'verb_switching', 'full_switching']
    for level in levels:
        create_corrupted_codeswitched_sva_dataset(noun_list_tuples, verb_list_tuples, pred_verb_tuples, 
                                                 codeswitch_level=level)

Saved corrupted dataset at: ./corrupted_cs_datasets/corrupted_spanish_no_switching_sva.json
Saved corrupted dataset at: ./corrupted_cs_datasets/corrupted_spanish_noun_switching_sva.json
Saved corrupted dataset at: ./corrupted_cs_datasets/corrupted_spanish_verb_switching_sva.json
Saved corrupted dataset at: ./corrupted_cs_datasets/corrupted_spanish_full_switching_sva.json


## rest

In [7]:
def read_files(model, file_name: str) -> list:
    '''
    Read file of tuples and get only words associated with
    a single token by the model's tokenizr.

    Args:
        model: The model used for tokenization.
        file_name: The name of the file to read.

    Returns:
        A list of valid tuples containing words associated with a single token.
    '''
    f = open(file_name, "r")
    list_str = f.read()
    words_list =list_str.split('\n')[1:]
    clean_pair_list = [words_pair[words_pair.find('.')+2:] for words_pair in words_list]
    #print(clean_pair_list)
    clean_pair_list_tuples = [(words_pair.split('/')[0].strip(), words_pair.split('/')[1].strip()) for words_pair in clean_pair_list]
    #print(clean_pair_list_tuples)
    # Get only valid tuples
    valid_tuples = []
    for pair_tuple in clean_pair_list_tuples:
        toks_sing = model.to_tokens(f' {pair_tuple[0]}')[0].shape[0]
        toks_plur = model.to_tokens(f' {pair_tuple[1]}')[0].shape[0]
        if toks_sing==2 and toks_plur==2:
            valid_tuples.append(pair_tuple)
    return valid_tuples

In [8]:
read_files(model, "word_lists/plausible_fr_singular_plural_past_verbs.txt")

NameError: name 'model' is not defined

In [ ]:
def get_valid_french_verbs_nouns(model):
    """Load valid singular/plural noun and verb pairs for French."""
    # Main verbs (être, avoir) and common nouns
    examples_valid_verbs_tuples_pred = [('est', 'sont'), ('était', 'étaient'), ('a', 'ont'), ('avait', 'avaient')]
    examples_valid_verbs_tuples = [('regardait', 'regardaient'), ('parlait', 'parlaient')]
    examples_valid_nouns = [('artiste', 'artistes'), ('ministre', 'ministres'), ('docteur', 'docteurs')]
    verb_list_tuples = list(set(examples_valid_verbs_tuples + read_files(model, "word_lists/plausible_fr_singular_plural_past_verbs.txt")))  # Will add + read_files() later
    noun_list_tuples = list(set(examples_valid_nouns + read_files(model, "word_lists/fr_singular_plural_nouns.txt")))  # Will add + read_files() later
    return verb_list_tuples, noun_list_tuples, examples_valid_verbs_tuples_pred

def get_valid_spanish_verbs_nouns(model):
    # Get suitable list of verbs and nouns in singular and plural
    examples_valid_verbs_tuples_pred = [('es', 'son'), ('era', 'eran'), ('fue', 'fueron'), ('tuvo', 'tuvieron'), ('tiene', 'tienen')]
    examples_valid_verbs_tuples = [('tuvo', 'tuvieron')]
    examples_valid_nouns = [('cantante', 'cantantes'), ('ingeniero', 'ingenieros'), ('ministro', 'ministros'), ('piloto', 'pilotos')]
    verb_list_tuples = list(set(examples_valid_verbs_tuples + read_files(model, "word_lists/plausible_spa_singular_plural_past_verbs.txt")))
    noun_list_tuples = list(set(examples_valid_nouns + read_files(model, "datasets/word_lists/spa_singular_plural_nouns.txt")))
    return verb_list_tuples, noun_list_tuples, examples_valid_verbs_tuples_pred


In [ ]:
def create_dataset(model, language):
    """
    Create a dataset for subject-verb agreement (SVA) in Spanish or French.

    Args:
        model: The language model to extract valid verbs and nouns.
        language (str): "spanish" or "french" to determine sentence structure and grammar.

    Returns:
        Saves the dataset as JSON files split into train, validation, and test sets.
    """

    if language == "spanish":
        verb_list_tuples, noun_list_tuples, examples_valid_verbs_tuples_pred = get_valid_spanish_verbs_nouns(model)
    elif language == "french":
        verb_list_tuples, noun_list_tuples, examples_valid_verbs_tuples_pred = get_valid_french_verbs_nouns(model)
    else:
        raise ValueError("Unsupported language. Choose 'spanish' or 'french'.")
    print(verb_list_tuples)
    print(noun_list_tuples)
    print(examples_valid_verbs_tuples_pred)

    noun_list_sing = [f' {noun_tuple[0]}' for noun_tuple in noun_list_tuples]
    noun_list_plural = [f' {noun_tuple[1]}' for noun_tuple in noun_list_tuples]

    verb_1_list_sing = [f' {verb_tuple[0]}' for verb_tuple in verb_list_tuples]
    verb_1_list_plural = [f' {verb_tuple[1]}' for verb_tuple in verb_list_tuples]

    verb_2_list_sing = [f' {verb_tuple[0]}' for verb_tuple in examples_valid_verbs_tuples_pred]
    verb_2_list_plural = [f' {verb_tuple[1]}' for verb_tuple in examples_valid_verbs_tuples_pred]

    # Generate permutations
    permutations_list = [[i, j, k] for i in range(len(noun_list_sing)) 
                                      for j in range(len(verb_1_list_sing)) 
                                      for k in range(len(noun_list_sing)) if k != i]
    permutations_array = np.array(permutations_list)
    
    np.random.seed(42)  # Set seed for reproducibility
    np.random.shuffle(permutations_array)

    final_dict = {}

    for valid_counter, (i, j, k) in enumerate(permutations_array):
        final_dict[valid_counter] = {}

        if language == "spanish":
            sent_1 = f'Los{noun_list_plural[k]} que{verb_1_list_plural[j]} al{noun_list_sing[i]}'
            sent_2 = f'El{noun_list_sing[k]} que{verb_1_list_sing[j]} al{noun_list_sing[i]}'
        elif language == "french":
            sent_1 = f'Des{noun_list_plural[k]} qui{verb_1_list_plural[j]} un{noun_list_sing[i]}'
            sent_2 = f'Un{noun_list_sing[k]} qui{verb_1_list_sing[j]} un{noun_list_sing[i]}'

        # Avoid verb repetition
        verbs_indices = list(range(len(verb_2_list_sing)))
        already_verb = j
        available_verb_indices = verbs_indices[:already_verb] + verbs_indices[already_verb+1:]
        ver_2_idx = np.random.choice(available_verb_indices)

        # Randomly decide base/singular structure
        rdm_num = int(round(random.uniform(0, 1), 0))

        if rdm_num == 0:
            src = sent_1
            base = sent_2
            src_label = verb_2_list_plural[ver_2_idx]
            base_label = verb_2_list_sing[ver_2_idx]
        else:
            src = sent_2
            base = sent_1
            src_label = verb_2_list_sing[ver_2_idx]
            base_label = verb_2_list_plural[ver_2_idx]
        
        final_dict[valid_counter]['src'] = src
        final_dict[valid_counter]['base'] = base
        final_dict[valid_counter]['src_label'] = src_label
        final_dict[valid_counter]['base_label'] = base_label
        

        # Determine subject number based on verb conjugation
        if base_label.split()[-1].endswith(('n', 'ent')):  # Spanish plural ends in 'n', French often in 'ent'
            final_dict[valid_counter]['base_subject_number'] = 'plural'
        else:
            final_dict[valid_counter]['base_subject_number'] = 'singular'

    # Split dataset into train, validation, and test sets
    len_dataset = len(final_dict)
    train_end_idx = len_dataset * 70 // 100
    val_end_idx = len_dataset * 85 // 100
    split_dicts = {
        'train': {k: final_dict[k] for k in list(final_dict)[:train_end_idx]},
        'validation': {k: final_dict[k] for k in list(final_dict)[train_end_idx:val_end_idx]},
        'test': {k: final_dict[k] for k in list(final_dict)[val_end_idx:]}
    }

    for split in ['train', 'validation', 'test']:
        final_datasets_dir = 'final_datasets'
        os.makedirs(final_datasets_dir, exist_ok=True)

        output_path = f'{final_datasets_dir}/{language}_{split}_sva_dataset.json'
        with open(output_path, 'w') as f:
            json.dump(split_dicts[split], f, indent=4)

        print(f"Dataset saved to {output_path}")

In [ ]:
create_dataset(model=model, language="french")

[('parlait', 'parlaient'), ('regardait', 'regardaient'), ('avait', 'avaient')]
[('artiste', 'artistes'), ('consul', 'consuls'), ('garde', 'gardes'), ('docteur', 'docteurs'), ('consultant', 'consultants'), ('prince', 'princes'), ('roi', 'rois'), ('assistant', 'assistants'), ('journaliste', 'journalistes'), ('ministre', 'ministres'), ('producteur', 'producteurs'), ('employé', 'employés'), ('vendeur', 'vendeurs'), ('chef', 'chefs'), ('designer', 'designers'), ('prêtre', 'prêtres'), ('député', 'députés'), ('cadet', 'cadets'), ('acteur', 'acteurs'), ('associé', 'associés'), ('scientifique', 'scientifiques'), ('maître', 'maîtres'), ('pilote', 'pilotes'), ('marin', 'marins'), ('agent', 'agents')]
[('est', 'sont'), ('était', 'étaient'), ('a', 'ont'), ('avait', 'avaient')]
Dataset saved to final_datasets/french_train_sva_dataset.json
Dataset saved to final_datasets/french_validation_sva_dataset.json
Dataset saved to final_datasets/french_test_sva_dataset.json


Checking single token verbs

In [ ]:
#model, language, subject_number='both', split='train', num_samples=100
language='french'
split='train'
num_samples=100
subject_number='both'

dataset_path = 'final_datasets'  # Set to the directory containing the dataset file

# Construct the filename based on language and split
filename = f'{language}_{split}_sva_dataset.json'
file_path = os.path.join(dataset_path, filename)

# Load the dataset from the JSON file
with open(file_path, 'r', encoding='utf-8') as file:
    data = json.load(file)

# Initialize lists to store the data
base_list = []
src_list = []
base_label_list = []
src_label_list = []
answers = []
ex_lang_list = []
ex_number_list = []

# Process the loaded data
for i, item in enumerate(data.values()):
    if i < num_samples:
        if subject_number.lower() != 'both':
            if subject_number.lower() == item['base_subject_number'].lower():
                base_list.append(item['base'])
                src_list.append(item['src'])
                base_label_list.append(item['base_label'])
                src_label_list.append(item['src_label'])
                answers.append((item['base_label'], item['src_label']))
                ex_lang_list.append(language)
                ex_number_list.append(item['base_subject_number'])
    
        else:
            # assert num_samples % 2 == 0 , "num_samples must be even"
            base_list.append(item['base'])
            src_list.append(item['src'])
            base_label_list.append(item['base_label'])
            src_label_list.append(item['src_label'])
            answers.append((item['base_label'], item['src_label']))
            ex_lang_list.append(language)
            ex_number_list.append(item['base_subject_number'])

print((np.array(ex_number_list)=='singular').sum())
print((np.array(ex_number_list)=='plural').sum())
print({'base_list': base_list,
        'src_list': src_list,
        'base_label_list': base_label_list,
        'src_label_list': src_label_list,
        'answers': answers,
        'ex_lang_list': ex_lang_list,
        'ex_number_list': ex_number_list})

72
28
{'base_list': ['Un assistant qui parlait un pilote', 'Un assistant qui parlait un chef', 'Un prince qui parlait un vendeur', 'Un associé qui parlait un garde', 'Un agent qui regardait un cadet', 'Des marins qui avaient un artiste', 'Des journalistes qui regardaient un docteur', 'Des employés qui regardaient un consultant', 'Un docteur qui parlait un marin', 'Un maître qui avait un chef', 'Un roi qui parlait un marin', 'Des pilotes qui regardaient un vendeur', 'Un consul qui parlait un cadet', 'Un designer qui regardait un vendeur', 'Des députés qui parlaient un journaliste', 'Des consuls qui avaient un cadet', 'Des gardes qui avaient un assistant', 'Des princes qui avaient un designer', 'Un consul qui regardait un assistant', 'Des prêtres qui avaient un marin', 'Un consul qui parlait un producteur', 'Un producteur qui avait un assistant', 'Des employés qui regardaient un assistant', 'Un prince qui regardait un cadet', 'Des députés qui avaient un roi', 'Un garde qui parlait un min

In [ ]:
import torch

examples_valid_verbs_tuples_pred = read_files(model, "word_lists/plausible_fr_singular_plural_past_verbs.txt")

def check_single_token_verbs(model, verb_pairs):
    valid_verbs = []
    
    for singular, plural in verb_pairs:
        try:
            sing_token = model.to_single_token(singular)
            plur_token = model.to_single_token(plural)
            if (singular, plural) not in valid_verbs:
                valid_verbs.append((singular, plural))
        except AssertionError:
            pass  # Skip multi-token words

    print("Valid single-token verbs:")
    for verb in valid_verbs:
        print(verb)

    return valid_verbs

# Run the check
filtered_valid_verbs = check_single_token_verbs(model, examples_valid_verbs_tuples_pred)


NameError: name 'read_files' is not defined

# corrupted experiments:

In [1]:
import json
import numpy as np
import torch
from sklearn.decomposition import PCA
import plotly.express as px
from transformer_lens import HookedTransformer
import os
import time
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# === Logging helper ===
def log(msg):
    print(f"[{time.strftime('%Y-%m-%d %H:%M:%S')}] {msg}", flush=True)

# === Load model ===
log("Loading model...")
model = HookedTransformer.from_pretrained("gemma-2b")
model.set_use_attn_result(True)
log("Model loaded.")

# === Experiment setup ===
levels = ['no_switching', 'noun_switching', 'verb_switching', 'full_switching']
data_dir = './datasets/new_exp'
figs = []

[2025-04-16 03:14:21] Loading model...


`config.hidden_act` is ignored, you should use `config.hidden_activation` instead.
Gemma's activation function will be set to `gelu_pytorch_tanh`. Please, use
`config.hidden_activation` if you want to override this behaviour.
See https://github.com/huggingface/transformers/pull/29402 for more details.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Loaded pretrained model gemma-2b into HookedTransformer
[2025-04-16 03:14:43] Model loaded.


In [11]:
# === Experiment setup ===
levels = ['no_switching', 'noun_switching', 'verb_switching', 'full_switching']
data_dir = '../datasets/corrupted_cs_datasets'  # Path to corrupted datasets
figs = []

In [14]:
# === Run PCA & visualize each switching level with corrupted data ===
for level in levels:
    log(f"Processing corrupted level: {level}")
    dataset_path = f"{data_dir}/corrupted_spanish_{level}_sva.json"
    
    with open(dataset_path, 'r', encoding='utf-8') as file:
        dataset = json.load(file)

    sentences, labels = [], []
    for item in dataset.values():
        sentences.append(item['src'])
        labels.append(item['base_subject_number'])

    log(f"Tokenizing {len(sentences)} sentences...")
    tokens = model.to_tokens(sentences)

    log("Running model and extracting attention head activations...")
    # Using the same layer (13) and head (7) as in the original file
    logits, cache = model.run_with_cache(tokens, names_filter=["blocks.13.attn.hook_result"])
    activations = cache["blocks.13.attn.hook_result"][:, -1, 7, :].cpu().numpy()

    log("Running PCA...")
    pca = PCA(n_components=2)
    projected = pca.fit_transform(activations)

    log("Generating plot...")
    fig = px.scatter(
        x=projected[:, 0], y=projected[:, 1],
        color=labels,
        labels={'x': 'PC1', 'y': 'PC2'},
        title=f"Corrupted Codeswitching: {level}",
        hover_name=sentences
    )
    figs.append(fig)

# === Combine all plots into a single subplot ===
log("Combining plots into subplot layout...")
fig_subplots = make_subplots(
    rows=1, cols=4, 
    subplot_titles=[f"Corrupted {level}" for level in levels]
)

for i, fig in enumerate(figs, start=1):
    for trace in fig.data:
        fig_subplots.add_trace(trace, row=1, col=i)

# Update layout for better visualization
fig_subplots.update_layout(
    height=600, 
    width=2000, 
    title_text="PCA of Corrupted Spanish Codeswitched Datasets",
    showlegend=True,
    legend=dict(
        title="Subject Number",
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1
    )
)

# Update x and y axes labels
for i in range(1, 5):
    fig_subplots.update_xaxes(title_text="PC1", row=1, col=i)
    fig_subplots.update_yaxes(title_text="PC2", row=1, col=i)

# === Save plots ===
output_dir = "corrupted_analysis"
os.makedirs(output_dir, exist_ok=True)
output_html = f"{output_dir}/corrupted_codeswitching_levels.html"
output_png = f"{output_dir}/corrupted_codeswitching_levels.png"

log(f"Saving to {output_html} and {output_png}...")
fig_subplots.write_html(output_html)
fig_subplots.write_image(output_png)
log("Done.")

# === Optional display (uncomment to show in notebook) ===
# fig_subplots.show()

[2025-04-16 03:41:59] Processing corrupted level: no_switching
[2025-04-16 03:41:59] Tokenizing 120 sentences...
[2025-04-16 03:41:59] Running model and extracting attention head activations...
[2025-04-16 03:42:00] Running PCA...
[2025-04-16 03:42:00] Generating plot...
[2025-04-16 03:42:00] Processing corrupted level: noun_switching
[2025-04-16 03:42:00] Tokenizing 120 sentences...
[2025-04-16 03:42:00] Running model and extracting attention head activations...
[2025-04-16 03:42:00] Running PCA...
[2025-04-16 03:42:00] Generating plot...
[2025-04-16 03:42:00] Processing corrupted level: verb_switching
[2025-04-16 03:42:00] Tokenizing 120 sentences...
[2025-04-16 03:42:00] Running model and extracting attention head activations...
[2025-04-16 03:42:00] Running PCA...
[2025-04-16 03:42:00] Generating plot...
[2025-04-16 03:42:00] Processing corrupted level: full_switching
[2025-04-16 03:42:00] Tokenizing 120 sentences...
[2025-04-16 03:42:00] Running model and extracting attention head

Exception: The (row, col) pair sent is out of range. Use Figure.print_grid to view the subplot grid. 

## Accuracy experiment:

In [20]:
import os
import json
import time
import torch
import numpy as np
from collections import defaultdict
import plotly.express as px
from transformer_lens import HookedTransformer, utils
from load_dataset import load_sva_dataset


def log(msg):
    print(f"[{time.strftime('%Y-%m-%d %H:%M:%S')}] {msg}", flush=True)


def evaluate_logit_preference(model, dataset):
    correct = 0
    total = 0

    for i in range(len(dataset['src_list'])):
        sentence = dataset['src_list'][i]
        correct_label = dataset['src_label_list'][i]
        incorrect_label = dataset['base_label_list'][i]
        tokens = model.to_tokens(sentence).to(model.cfg.device)

        with torch.no_grad():
            logits = model(tokens)
            final_logits = logits[0, -1]  # logits for the last token

        try:
            correct_id = model.to_single_token(correct_label)
            incorrect_id = model.to_single_token(incorrect_label)
        except AssertionError:
            continue  # skip if either is not a single token

        if final_logits[correct_id] > final_logits[incorrect_id]:
            correct += 1
        total += 1

    return {
        'accuracy': correct / total,
        'total': total
    }


def main():
    log("Loading model...")
    model = HookedTransformer.from_pretrained("gemma-2b")
    model.set_use_attn_result(True)
    model.to(utils.get_device())
    log("Model loaded.")

    levels = ['no_switching', 'noun_switching', 'verb_switching', 'full_switching']
    results = defaultdict(dict)

    for level in levels:
        log(f"Evaluating codeswitch level: {level}")
        dataset_path = f"../datasets/new_exp/spanish_{level}_sva.json"
        with open(dataset_path, 'r', encoding='utf-8') as f:
            raw_data = json.load(f)

        dataset = {
            'src_list': [ex['src'] for ex in raw_data.values()],
            'src_label_list': [ex['src_label'] for ex in raw_data.values()],
            'base_label_list': [ex['base_label'] for ex in raw_data.values()]
        }

        metrics = evaluate_logit_preference(model, dataset)
        results[level] = metrics
        log(f"{level} => Accuracy: {metrics['accuracy']:.2%}")

    # Plotting
    levels = list(results.keys())
    accuracies = [results[l]['accuracy'] for l in levels]

    fig = px.bar(
        x=levels,
        y=accuracies,
        labels={'x': 'Code-switching Level', 'y': 'Accuracy'},
        title="Logit Preference Accuracy on SVA Task Under Code-switching"
    )

    os.makedirs("images/accuracy", exist_ok=True)
    fig.write_html("images/accuracy/sva_accuracy_logit_preference.html")
    fig.write_image("images/accuracy/sva_accuracy_logit_preference.png")
    log("Saved accuracy plots.")


if __name__ == "__main__":
    main()


[2025-04-16 04:15:11] Loading model...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loaded pretrained model gemma-2b into HookedTransformer
Moving model to device:  cuda
[2025-04-16 04:15:29] Model loaded.
[2025-04-16 04:15:29] Evaluating codeswitch level: no_switching
[2025-04-16 04:15:34] no_switching => Accuracy: 77.50%
[2025-04-16 04:15:34] Evaluating codeswitch level: noun_switching
[2025-04-16 04:15:39] noun_switching => Accuracy: 76.25%
[2025-04-16 04:15:39] Evaluating codeswitch level: verb_switching
[2025-04-16 04:15:44] verb_switching => Accuracy: 87.50%
[2025-04-16 04:15:44] Evaluating codeswitch level: full_switching
[2025-04-16 04:15:50] full_switching => Accuracy: 81.25%
[2025-04-16 04:15:50] Saved accuracy plots.
